# Phase 2 — Pre-release hit classification

Uses the feature set built and validated in `04_phase2_feature_engineering.ipynb`.

**Question:** can we predict whether a game will be a commercial hit (≥1.0M units) using only
information available before it ships?

**Why classification rather than regression:** Phase 1's metrics were unstable because
`Global_Sales` is heavily right-skewed — removing a single row (Wii Sports) moved R² by 0.14.
Classification doesn't square an 82.53M error, so a blockbuster carries the same weight as any
other hit, and the reported numbers stop depending on which extreme rows landed in which split.

**Three leakage vectors are controlled here:**
1. *Arithmetic* (the parent paper's): no regional sales columns anywhere.
2. *Temporal / post-outcome*: no `User_Count`, `Critic_Count`, or `User_Score`; all history
   features use strictly-earlier-year data (see notebook 04).
3. *Duplicate-title*: the dataset is one row per game-**platform** release, so a multi-platform
   title appears several times with near-identical features. Splits are grouped on `Name`, so
   every platform variant of a game stays in the same fold — otherwise the model would be
   scored on games it had already trained on.

In [1]:
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, roc_auc_score, roc_curve,
    classification_report,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

data = pd.read_csv("phase2_features.csv")
print(f"{len(data)} rows | hit rate {data['is_hit'].mean():.1%}")

16416 rows | hit rate 12.5%


## 1. Missing-value strategy

`prior_franchise_mean_sales` is populated for only ~47% of rows — the rest are the first
detected entry in their franchise. Dropping them would discard every new IP and bias the model
toward established franchises, so instead: add an explicit **`has_franchise_history` indicator**
and fill the history columns with 0. The indicator keeps "no track record" distinguishable from
"a track record of zero sales", which a bare 0-fill would conflate.

In [2]:
data["has_franchise_history"] = data["prior_franchise_mean_sales"].notna().astype(int)
data["has_publisher_history"] = data["prior_publisher_mean_sales"].notna().astype(int)

for c in ["prior_franchise_mean_sales", "prior_franchise_max_sales",
          "prior_publisher_mean_sales", "prior_publisher_max_sales"]:
    data[c] = data[c].fillna(0.0)

data["Rating"] = data["Rating"].fillna("Unknown")
data["genre_density_prior_year"] = data["genre_density_prior_year"].fillna(0)

TOP_N_PUBLISHERS = 20
top_pubs = data["Publisher"].value_counts().head(TOP_N_PUBLISHERS).index
data["Publisher_bucketed"] = data["Publisher"].where(data["Publisher"].isin(top_pubs), "Other")

print(f"has_franchise_history: {data['has_franchise_history'].mean():.1%} of rows")
print(f"has_publisher_history: {data['has_publisher_history'].mean():.1%} of rows")

has_franchise_history: 47.4% of rows
has_publisher_history: 94.5% of rows


## 2. Feature assembly

**Tier 1** = announcement-time only. **Tier 2** = Tier 1 + `Critic_Score` (available at launch
when review embargoes lift). Tier 2 is evaluated separately in Section 6.

In [3]:
NUMERIC_T1 = [
    "Year_of_Release", "is_sequel", "has_franchise_history", "has_publisher_history",
    "prior_franchise_count", "prior_franchise_mean_sales", "prior_franchise_max_sales",
    "prior_publisher_count", "prior_publisher_mean_sales", "prior_publisher_max_sales",
    "simultaneous_platforms", "platform_age", "genre_density_prior_year",
]
CATEGORICAL = ["Genre", "Platform", "Rating", "Publisher_bucketed"]


def build_X(d: pd.DataFrame, numeric_cols) -> pd.DataFrame:
    cat = pd.get_dummies(d[CATEGORICAL], drop_first=True)
    return pd.concat([d[numeric_cols].reset_index(drop=True), cat.reset_index(drop=True)], axis=1)


X = build_X(data, NUMERIC_T1)
y = data["is_hit"].values
groups = data["Name"].values

print(f"Tier 1 feature matrix: {X.shape[0]} rows x {X.shape[1]} columns")
print(f"Distinct title groups: {pd.Series(groups).nunique()}")

Tier 1 feature matrix: 16416 rows x 82 columns
Distinct title groups: 11397


## 3. Models and evaluation protocol

5-fold `StratifiedGroupKFold` — grouped on `Name` (no title straddles folds) and stratified on
the target (each fold keeps the ~12.5% hit rate).

Both models use `class_weight="balanced"` because the classes are imbalanced. Metrics:
- **ROC-AUC** — threshold-independent ranking quality. Baseline for a useless model: 0.50.
- **Average precision (PR-AUC)** — the more honest metric under imbalance. Baseline: 0.125,
  the base rate.
- **Precision / recall / F1** at the default 0.5 threshold.

Accuracy is deliberately not reported as a headline: always predicting "not a hit" scores 87.5%
accuracy while being useless.

In [4]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, max_depth=12, min_samples_leaf=3,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

SCORING = ["roc_auc", "average_precision", "precision", "recall", "f1"]

rows = []
for name, model in models.items():
    s = cross_validate(model, X, y, cv=cv, groups=groups, scoring=SCORING, n_jobs=1)
    rows.append({
        "model": name,
        "ROC_AUC": s["test_roc_auc"].mean(), "ROC_AUC_std": s["test_roc_auc"].std(),
        "Avg_Precision": s["test_average_precision"].mean(),
        "Precision": s["test_precision"].mean(),
        "Recall": s["test_recall"].mean(),
        "F1": s["test_f1"].mean(),
    })

tier1_summary = pd.DataFrame(rows)
tier1_summary.to_csv(os.path.join(OUT_DIR, "phase2_tier1_cv_metrics.csv"), index=False)
print(f"Baselines -> ROC_AUC 0.500 | Avg_Precision {y.mean():.3f} (base rate)\n")
tier1_summary.round(4)

Baselines -> ROC_AUC 0.500 | Avg_Precision 0.125 (base rate)



,model,ROC_AUC,ROC_AUC_std,Avg_Precision,Precision,Recall,F1
0,Logistic Regression,0.8441,0.0091,0.4778,0.3093,0.7656,0.4405
1,Random Forest,0.8644,0.0120,0.5316,0.3924,0.6880,0.4996


## 4. Out-of-fold ROC and precision-recall curves

Each title is predicted by a fold that never saw it (nor any of its other platform releases).

In [5]:
best_name = tier1_summary.sort_values("ROC_AUC", ascending=False).iloc[0]["model"]
best_model = models[best_name]
print(f"Best Tier 1 model by ROC-AUC: {best_name}")

oof_proba = cross_val_predict(best_model, X, y, cv=cv, groups=groups, method="predict_proba", n_jobs=1)[:, 1]
print(f"Out-of-fold ROC-AUC: {roc_auc_score(y, oof_proba):.4f}")
print(f"Out-of-fold Avg precision: {average_precision_score(y, oof_proba):.4f}  (base rate {y.mean():.4f})")

fpr, tpr, _ = roc_curve(y, oof_proba)
prec, rec, _ = precision_recall_curve(y, oof_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(fpr, tpr, color="#4C72B0", label=f"AUC = {roc_auc_score(y, oof_proba):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="chance")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title(f"ROC - {best_name} (out-of-fold)"); axes[0].legend()

axes[1].plot(rec, prec, color="#C44E52", label=f"AP = {average_precision_score(y, oof_proba):.3f}")
axes[1].axhline(y.mean(), ls="--", c="k", lw=1, label=f"base rate = {y.mean():.3f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title(f"Precision-Recall - {best_name}"); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "phase2_roc_pr_curves.png"), dpi=150)
plt.close()
print("Saved outputs/phase2_roc_pr_curves.png")

Best Tier 1 model by ROC-AUC: Random Forest


Out-of-fold ROC-AUC: 0.8639
Out-of-fold Avg precision: 0.5282  (base rate 0.1250)


Saved outputs/phase2_roc_pr_curves.png


In [6]:
print(classification_report(y, (oof_proba >= 0.5).astype(int), target_names=["not hit", "hit"], digits=3))

              precision    recall  f1-score   support

     not hit      0.950     0.848     0.896     14364
         hit      0.392     0.687     0.500      2052

    accuracy                          0.828     16416
   macro avg      0.671     0.768     0.698     16416
weighted avg      0.880     0.828     0.847     16416



## 5. What drives a pre-release hit prediction?

In [7]:
rf = RandomForestClassifier(
    n_estimators=400, max_depth=12, min_samples_leaf=3,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
).fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 7))
importances.head(18)[::-1].plot(kind="barh", color="#55A868")
plt.title("Random Forest feature importance - pre-release hit prediction")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "phase2_feature_importance.png"), dpi=150)
plt.close()

importances.head(15)

prior_publisher_mean_sales            0.156381
prior_franchise_mean_sales            0.117032
prior_publisher_max_sales             0.099549
prior_franchise_max_sales             0.097900
Year_of_Release                       0.054874
genre_density_prior_year              0.045338
Publisher_bucketed_Other              0.042318
platform_age                          0.042179
prior_publisher_count                 0.042072
Publisher_bucketed_Nintendo           0.031588
prior_franchise_count                 0.022485
simultaneous_platforms                0.019641
Rating_Unknown                        0.014280
Rating_M                              0.014145
Publisher_bucketed_Electronic Arts    0.010444
dtype: float64

## 6. Does launch-day critic score add anything? (Tier 1 vs Tier 2)

Critically, both tiers are evaluated on the **same 7,982 rows** — those with a `Critic_Score`.
Comparing Tier 1 on 16,416 rows against Tier 2 on 7,982 would confound the feature change with
a sample change.

In [8]:
sub = data[data["Critic_Score"].notna()].reset_index(drop=True)
y_sub = sub["is_hit"].values
groups_sub = sub["Name"].values
print(f"Shared comparison subset: {len(sub)} rows | hit rate {y_sub.mean():.1%}")

X_t1 = build_X(sub, NUMERIC_T1)
X_t2 = build_X(sub, NUMERIC_T1 + ["Critic_Score"])

tier_rows = []
for tier_name, Xt in [("Tier 1 (announcement only)", X_t1), ("Tier 2 (+ Critic_Score)", X_t2)]:
    for model_name, model in models.items():
        s = cross_validate(model, Xt, y_sub, cv=cv, groups=groups_sub,
                           scoring=["roc_auc", "average_precision"], n_jobs=1)
        tier_rows.append({
            "feature_set": tier_name,
            "model": model_name,
            "ROC_AUC": s["test_roc_auc"].mean(),
            "Avg_Precision": s["test_average_precision"].mean(),
        })

tier_comparison = pd.DataFrame(tier_rows)
tier_comparison.to_csv(os.path.join(OUT_DIR, "phase2_tier1_vs_tier2.csv"), index=False)
tier_comparison.round(4)

Shared comparison subset: 7982 rows | hit rate 16.8%


,feature_set,model,ROC_AUC,Avg_Precision
0,Tier 1 (announcement only),Logistic Regression,0.8073,0.4853
1,Tier 1 (announcement only),Random Forest,0.8332,0.5414
2,Tier 2 (+ Critic_Score),Logistic Regression,0.8599,0.6001
3,Tier 2 (+ Critic_Score),Random Forest,0.8768,0.6481


## 7. Robustness: chronological split

Random CV lets the model train on 2016 titles to predict 2005 ones — which never happens in
practice. A chronological split simulates real deployment: train on everything through 2013,
predict 2014-2016. Expect lower scores; the gap between this and random CV is itself the
finding, since it measures how much performance depends on hindsight.

In [9]:
train_mask = data["Year_of_Release"] <= 2013
test_mask = data["Year_of_Release"] >= 2014

X_tr, y_tr = X[train_mask.values], y[train_mask.values]
X_te, y_te = X[test_mask.values], y[test_mask.values]
print(f"Train (<=2013): {len(y_tr)} rows, hit rate {y_tr.mean():.1%}")
print(f"Test  (>=2014): {len(y_te)} rows, hit rate {y_te.mean():.1%}")

chrono_rows = []
for name, model in models.items():
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    chrono_rows.append({
        "model": name,
        "ROC_AUC_chronological": roc_auc_score(y_te, proba),
        "Avg_Precision_chronological": average_precision_score(y_te, proba),
    })

chrono = pd.DataFrame(chrono_rows).merge(
    tier1_summary[["model", "ROC_AUC", "Avg_Precision"]].rename(
        columns={"ROC_AUC": "ROC_AUC_randomCV", "Avg_Precision": "Avg_Precision_randomCV"}),
    on="model",
)
chrono.to_csv(os.path.join(OUT_DIR, "phase2_chronological_split.csv"), index=False)
chrono.round(4)

Train (<=2013): 14725 rows, hit rate 12.8%
Test  (>=2014): 1691 rows, hit rate 10.1%


,model,ROC_AUC_chronological,Avg_Precision_chronological,ROC_AUC_randomCV,Avg_Precision_randomCV
0,Logistic Regression,0.8816,0.4572,0.8441,0.4778
1,Random Forest,0.8836,0.5111,0.8644,0.5316


## Summary

Read the tables above for the numbers. The structural points for the write-up:

- The target is **pre-release hit classification** (≥1.0M units, 12.5% base rate), evaluated
  with ROC-AUC and average precision against explicit baselines (0.50 and 0.125).
- Every feature is knowable before the game ships. No regional sales, no post-release review
  counts, and all franchise/publisher history restricted to strictly earlier years.
- Splits are grouped by title, so multi-platform releases of the same game cannot leak across
  folds — a third leakage vector the parent paper's setup would not have caught.
- The Tier 1 vs Tier 2 comparison (same rows) isolates whether launch-day critic reception adds
  predictive value beyond franchise and publisher pedigree.
- The chronological split measures how much of the random-CV performance depends on hindsight.

Compared with Phase 1, this trades a headline number that looked larger for one that means
what it claims: a forecast made from information that genuinely exists at decision time.